# APEX live demo - Colab notebook

*Zero-install try-it-yourself path for IBM SkillsBuild AI Builders Challenge May 2026 judges.*

**Project:** APEX, an AI race engineer for adaptive racers, veteran-team drivers, and grassroots competitors. Built on IBM Granite.

**Repo:** https://github.com/StephenSook/apex (Apache 2.0)

**Submission deadline:** 2026-05-31, 11:59 PM ET.

## What this notebook does

Runs the entire APEX coaching pipeline in a free Google Colab cell. Three input slots (50 Hz telemetry CSV, FIA Certificate of Adaptations PDF, written debrief) into a corner-by-corner coaching report with a Granite Guardian audit verdict + provenance footer.

The pipeline:

1. **Granite-Docling 258M** parses the COA PDF into structured JSON.
2. **Granite Vision 4.1 4B** parses the timing-sheet PDF into a CSV (optional input).
3. **Granite TimeSeries TTM r2.1** (frozen, pretrained) forecasts next-session mini-sector envelopes from the telemetry CSV.
4. **CvxpyLayer QP physics projection** corrects every forecast step against the friction ellipse + bicycle model + jerk bound + COA simultaneity gate, producing a structured physics-violation log.
5. **Granite Guardian 4.1 8B** audits the violation log under a BYOC custom-rule schema and returns approve / flag / reject + reasoning trace.
6. **Granite 4.1 8B Instruct** narrates the coaching report in race-engineer voice with citations back into the COA + FIA Appendix L.

Reproducibility metadata footer on every report carries the Granite model versions, input file hashes, and the git SHA of the code that produced it.

## How to run

Choose `Runtime` -> `Run all`. Cold start on a free T4 GPU is ~90 seconds for the first model load. Subsequent runs reuse the cached weights.

Day 9 cutover: this notebook hits the Hugging Face Space backend (`https://huggingface.co/spaces/StephenSook/apex`). Until then, the cells below ship a canned Sarah Reynolds Donington Park lap-17 mock so the layout is reviewable end-to-end.

## 1. Install dependencies

Day 9 swap: this cell pins to `granite-tsfm`, `granite-guardian`, `langchain-ibm`, `cvxpylayers`, `docling`, and the rest of the Vinh-lane requirements once `app/backend/requirements.txt` ships.

In [ ]:
# Day 9 cutover: pin actual versions from app/backend/requirements.txt once Vinh ships.
# Until then, the mock cells below do not require the full Granite stack.

# !pip install --quiet granite-tsfm==0.2.* granite-guardian==0.1.* docling==1.* cvxpylayers==0.1.* langchain-ibm==0.1.*

## 2. Inputs - Sarah Reynolds Donington Park lap 17 fixture

Day 9 swap: file pickers for telemetry CSV + COA PDF + debrief textarea. Until then, the canned fixture below mirrors `fixtures/personas/sarah-reynolds-debrief.md` in the repo.

In [ ]:
driver_id = "sarah-reynolds-britcar-m240i"

debrief = (
    "Lost the rears mid-Old Hairpin again. Trail-brake on the hand lever "
    "is not as smooth as it was at Croft last month. Hairpin entry feels "
    "like the lever travel is too short. Sector 2 is leaving 0.34 seconds on "
    "the table per lap."
)

# Mock telemetry slice: 24 mini-sectors at 1 Hz, Sector 2 Old Hairpin slow.
import json

mock_inputs = {
    "driver_id": driver_id,
    "debrief": debrief,
    "telemetry_window_s": 24,
    "sample_rate_hz": 50,
    "vehicle": {"make": "BMW", "model": "M240i", "series": "Britcar Trophy 2026"},
    "coa_section_refs": ["Appendix L", "Adaptive-equipment provisions (synthetic fixture)"],
}
print(json.dumps(mock_inputs, indent=2))

## 3. Pipeline execution (mocked until Day 9 HF Space)

Day 9 swap: replaces this cell with `httpx.post('https://huggingface.co/spaces/StephenSook/apex/api/analyze', files=...)`. Until then, returns a canned `CoachingReport` shaped exactly to `app/shared/types.ts` so judges see the full layout today.

In [ ]:
import json

mock_report = {
    "driver_id": driver_id,
    "corners": [
        {
            "name": "Old Hairpin",
            "sector": 2,
            "current_delta_s": 0.34,
            "recommendation": (
                "Lengthen lever travel by 12 mm to recover trail-brake smoothness. "
                "the COA-derived c_overlap flag (from the hand-control hardware spec in the adaptive-equipment provisions of the synthetic COA) permits hand-control range adjustment between "
                "38 mm and 62 mm; current setup is at 26 mm."
            ),
            "citations": [
                {"fia_article": "Appendix L", "coa_section": "Adaptive-equipment provisions (synthetic fixture)"}
            ],
        },
        {
            "name": "McLeans",
            "sector": 1,
            "current_delta_s": 0.08,
            "recommendation": (
                "Brake earlier by 0.18 s but at 12 percent lower peak pressure. "
                "Forecast envelope shows a -0.06 s gain achievable inside COA "
                "simultaneity envelope."
            ),
            "citations": [
                {"fia_article": "Appendix L", "coa_section": "Adaptive-equipment provisions (synthetic fixture)"}
            ],
        },
        {
            "name": "Coppice",
            "sector": 3,
            "current_delta_s": -0.05,
            "recommendation": (
                "Already at PB benchmark. Hold throttle modulation as-is; "
                "do not change steering input cadence."
            ),
            "citations": [
                {"fia_article": "Appendix L", "coa_section": "Adaptive-equipment provisions (synthetic fixture)"}
            ],
        },
    ],
    "tuning_delta": {
        "parameter": "brake_lever_travel",
        "current": 26.0,
        "recommended": 38.0,
        "unit": "mm",
        "citation": {"fia_article": "Appendix L", "coa_section": "Adaptive-equipment provisions (synthetic fixture)"},
    },
    "forecast": [
        {"sector_idx": i, "mean": 0.30 - 0.012 * i, "low": 0.18 - 0.012 * i, "high": 0.42 - 0.012 * i}
        for i in range(24)
    ],
    "audit": {
        "verdict": "approve",
        "reasoning_trace": [
            "Friction ellipse OK across all 24 mini-sectors.",
            "Bicycle model OK; steering rate within hand-control envelope.",
            "COA-derived c_overlap flag honoured (derived from approved hand-control hardware specifications).",
            "Jerk bound OK; lever-travel suggestion within COA range.",
        ],
        "audit_id": "audit-demo-20260521-001",
    },
    "provenance": {
        "model_versions": {
            "granite_docling": "258M-v0.4",
            "granite_vision": "4.1-4B",
            "granite_ttm": "r2.1",
            "granite_instruct": "4.1-8B-Instruct",
            "granite_guardian": "4.1-8B",
        },
        "commit_sha": "see github.com/StephenSook/apex",
        "generated_at_iso": "2026-05-21T17:00:00Z",
    },
}

print(json.dumps(mock_report, indent=2))

## 4. Coaching report (rendered)

Same content as the `/analyze` web demo, rendered in markdown for browser readability.

In [ ]:
from IPython.display import Markdown

def render_report(report):
    lines = ["## Coaching report\n"]
    for corner in report["corners"]:
        delta = corner["current_delta_s"]
        sign = "+" if delta >= 0 else ""
        lines.append(f"### Sector {corner['sector']} - {corner['name']} ({sign}{delta:.2f}s)")
        lines.append(corner["recommendation"])
        for cite in corner["citations"]:
            lines.append(f"- Cited: {cite['fia_article']} via {cite['coa_section']}.")
        lines.append("")
    tuning = report["tuning_delta"]
    lines.append(f"### Tuning delta - {tuning['parameter']}")
    lines.append(
        f"{tuning['current']:.0f} {tuning['unit']} -> {tuning['recommended']:.0f} {tuning['unit']} "
        f"(cited: {tuning['citation']['fia_article']} via {tuning['citation']['coa_section']})."
    )
    audit = report["audit"]
    lines.append(f"\n### Granite Guardian verdict: **{audit['verdict'].upper()}**")
    for step in audit["reasoning_trace"]:
        lines.append(f"- {step}")
    lines.append(f"\nAudit id `{audit['audit_id']}`.")
    return "\n".join(lines)

Markdown(render_report(mock_report))

## 5. What ships before submission

- Day 9 (2026-05-28): cell 1 installs actual Granite versions; cell 3 calls live HF Space; cell 2 takes telemetry + COA + debrief from file pickers (`google.colab.files.upload()`).
- Day 11 (2026-05-30): `commit_sha` populates from CI; `generated_at_iso` from runtime.
- Day 12 (2026-05-31, ET): notebook URL appears in README + `/judges` page + BeMyApp submission form.

**Provenance footer** is the load-bearing trust signal for IBM judges. Every coaching claim cites a COA section + FIA Article. The Granite Guardian audit ID is replayable: re-pass the same audit_id through the live HF Space to retrieve the full reasoning trace.

*This notebook is part of the APEX submission for the IBM SkillsBuild AI Builders Challenge May 2026. License: Apache 2.0.*